<a href="https://colab.research.google.com/github/saisai257274/NLP-1/blob/main/2403A54111_NLP_Lab_Assignment_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 1: Text Similarity On Sample Data**

In [ ]:
import pandas as pd
data = pd.read_excel("/content/data.xlsx")

In [ ]:
display(data.head())

,Statements
0,I love watching anime
1,I love watching movies
2,I am learning Artificial Intelligence
3,I am learning Data Science


In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

def nltk_preprocessing_pipeline(text):
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    text = text.lower()

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokenized_words = word_tokenize(text)
    filtered_words = [word for word in tokenized_words if word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]
    clean_summary = ' '.join(lemmatized_words)

    return clean_summary

print("NLTK preprocessing pipeline function created successfully!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


NLTK preprocessing pipeline function created successfully!


In [ ]:
data['preprocessed_statements'] = data['Statements'].apply(nltk_preprocessing_pipeline)
display(data['preprocessed_statements'].head())

,preprocessed_statements
0,love watching anime
1,love watching movie
2,learning artificial intelligence
3,learning data science


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
doc_term_matrix = count_vectorizer.fit_transform(data['preprocessed_statements'])

In [ ]:
feature_names = count_vectorizer.get_feature_names_out()
bow_df = pd.DataFrame(doc_term_matrix.toarray(), columns=feature_names)

bow_top_10 = bow_df.head(10)
display(bow_top_10)

,anime,artificial,data,intelligence,learning,love,movie,science,watching
0,1,0,0,0,0,1,0,0,1
1,0,0,0,0,0,1,1,0,1
2,0,1,0,1,1,0,0,0,0
3,0,0,1,0,1,0,0,1,0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(bow_df)
cosine_sim_df = pd.DataFrame(cosine_sim_matrix, index=[data['preprocessed_statements']], columns=[data['preprocessed_statements']])

print("Cosine Similarity Matrix:")
print(cosine_sim_df)

Cosine Similarity Matrix:
preprocessed_statements          love watching anime love watching movie  \
preprocessed_statements                                                    
love watching anime                         1.000000            0.666667   
love watching movie                         0.666667            1.000000   
learning artificial intelligence            0.000000            0.000000   
learning data science                       0.000000            0.000000   

preprocessed_statements          learning artificial intelligence  \
preprocessed_statements                                             
love watching anime                                      0.000000   
love watching movie                                      0.000000   
learning artificial intelligence                         1.000000   
learning data science                                    0.333333   

preprocessed_statements          learning data science  
preprocessed_statements                      

In [ ]:
from sklearn.metrics import jaccard_score
import numpy as np

binary_bow = (bow_df > 0).astype(int)
num_documents = binary_bow.shape[0]
jaccard_sim_matrix = np.zeros((num_documents, num_documents))

for i in range(num_documents):
    for j in range(num_documents):
        vec_i = binary_bow.iloc[i]
        vec_j = binary_bow.iloc[j]

        intersection = np.sum(np.logical_and(vec_i, vec_j))
        union = np.sum(np.logical_or(vec_i, vec_j))

        if union == 0:
            jaccard_sim_matrix[i, j] = 0.0
        else:
            jaccard_sim_matrix[i, j] = intersection / union

df_jaccard_sim = pd.DataFrame(jaccard_sim_matrix, index=data['preprocessed_statements'], columns=data['preprocessed_statements'])

print("Jaccard Similarity Matrix:")
print(df_jaccard_sim)

Jaccard Similarity Matrix:
preprocessed_statements           love watching anime  love watching movie  \
preprocessed_statements                                                      
love watching anime                               1.0                  0.5   
love watching movie                               0.5                  1.0   
learning artificial intelligence                  0.0                  0.0   
learning data science                             0.0                  0.0   

preprocessed_statements           learning artificial intelligence  \
preprocessed_statements                                              
love watching anime                                            0.0   
love watching movie                                            0.0   
learning artificial intelligence                               1.0   
learning data science                                          0.2   

preprocessed_statements           learning data science  
preprocessed_statements  

In [ ]:
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

def document_wordnet_similarity(doc1, doc2):
    tokens1 = word_tokenize(doc1.lower())
    tokens2 = word_tokenize(doc2.lower())

    synsets1 = [s for token in tokens1 for s in wordnet.synsets(token)]
    synsets2 = [s for token in tokens2 for s in wordnet.synsets(token)]

    if not synsets1 or not synsets2:
        return 0.0

    max_similarities = []
    for s1 in synsets1:
        max_sim_for_s1 = 0.0
        for s2 in synsets2:
            sim = s1.path_similarity(s2)
            if sim is not None and sim > max_sim_for_s1:
                max_sim_for_s1 = sim
        max_similarities.append(max_sim_for_s1)

    if max_similarities:
        return np.mean(max_similarities)
    else:
        return 0.0

documents = data['preprocessed_statements']
num_documents = len(documents)
wordnet_sim_matrix = np.zeros((num_documents, num_documents))

for i in range(num_documents):
    for j in range(num_documents):
        if i == j:
            wordnet_sim_matrix[i, j] = 1.0
        else:
            wordnet_sim_matrix[i, j] = document_wordnet_similarity(documents[i], documents[j])

df_wordnet_sim = pd.DataFrame(wordnet_sim_matrix, index=documents, columns=documents)

print("WordNet Similarity Matrix (Path Similarity):")
print(df_wordnet_sim)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


WordNet Similarity Matrix (Path Similarity):
preprocessed_statements           love watching anime  love watching movie  \
preprocessed_statements                                                      
love watching anime                          1.000000             0.908712   
love watching movie                          0.953216             1.000000   
learning artificial intelligence             0.270461             0.270461   
learning data science                        0.277381             0.277381   

preprocessed_statements           learning artificial intelligence  \
preprocessed_statements                                              
love watching anime                                       0.241351   
love watching movie                                       0.251462   
learning artificial intelligence                          1.000000   
learning data science                                     0.736905   

preprocessed_statements           learning data science  
preproc

# **Task 2: Text Similarity On Assignment Data**

In [ ]:
import pandas as pd
df = pd.read_excel("/content/assignment_data.xlsx")

In [ ]:
display(df.head())

,Assignment_Statements
0,The sun dipped below the horizon as the city l...
1,A cup of coffee can sometimes solve half the d...
2,The library was unusually quiet that afternoon.
3,She decided to learn a new language this year.
4,Rainy days make long naps more enjoyable.


In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

def nltk_preprocessing_pipeline(text):
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    text = text.lower()

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokenized_words = word_tokenize(text)
    filtered_words = [word for word in tokenized_words if word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]
    clean_summary = ' '.join(lemmatized_words)

    return clean_summary

print("NLTK preprocessing pipeline function created successfully!")

NLTK preprocessing pipeline function created successfully!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
df['preprocessed_assignment_statements'] = df['Assignment_Statements'].apply(nltk_preprocessing_pipeline)
display(df['preprocessed_assignment_statements'].head())

,preprocessed_assignment_statements
0,sun dipped horizon city light flickered
1,cup coffee sometimes solve half day problem
2,library unusually quiet afternoon
3,decided learn new language year
4,rainy day make long nap enjoyable


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
doc_term_matrix = count_vectorizer.fit_transform(df['preprocessed_assignment_statements'])

In [ ]:
feature_names = count_vectorizer.get_feature_names_out()
bow_df = pd.DataFrame(doc_term_matrix.toarray(), columns=feature_names)

bow_top_10 = bow_df.head(10)
display(bow_top_10)

,act,afternoon,air,announced,arrived,bed,boost,car,change,city,...,time,today,tomorrow,train,tree,unexpected,unusually,waited,walk,year
0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
7,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
8,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(bow_df)
cosine_sim_df = pd.DataFrame(cosine_sim_matrix, index=[df['preprocessed_assignment_statements']], columns=[df['preprocessed_assignment_statements']])

print("Cosine Similarity Matrix:")
print(cosine_sim_df)

Cosine Similarity Matrix:
preprocessed_assignment_statements          sun dipped horizon city light flickered  \
preprocessed_assignment_statements                                                    
sun dipped horizon city light flickered                                         1.0   
cup coffee sometimes solve half day problem                                     0.0   
library unusually quiet afternoon                                               0.0   
decided learn new language year                                                 0.0   
rainy day make long nap enjoyable                                               0.0   
technology change faster people expect                                          0.0   
dog waited patiently door                                                       0.0   
early morning walk boost productivity                                           0.0   
forgot parked car                                                               0.0   
fresh air always 

In [ ]:
from sklearn.metrics import jaccard_score
import numpy as np

binary_bow = (bow_df > 0).astype(int)
num_documents = binary_bow.shape[0]
jaccard_sim_matrix = np.zeros((num_documents, num_documents))

for i in range(num_documents):
    for j in range(num_documents):
        vec_i = binary_bow.iloc[i]
        vec_j = binary_bow.iloc[j]

        intersection = np.sum(np.logical_and(vec_i, vec_j))
        union = np.sum(np.logical_or(vec_i, vec_j))

        if union == 0:
            jaccard_sim_matrix[i, j] = 0.0
        else:
            jaccard_sim_matrix[i, j] = intersection / union

df_jaccard_sim = pd.DataFrame(jaccard_sim_matrix, index=df['preprocessed_assignment_statements'], columns=df['preprocessed_assignment_statements'])

print("Jaccard Similarity Matrix:")
print(df_jaccard_sim)

Jaccard Similarity Matrix:
preprocessed_assignment_statements           sun dipped horizon city light flickered  \
preprocessed_assignment_statements                                                     
sun dipped horizon city light flickered                                          1.0   
cup coffee sometimes solve half day problem                                      0.0   
library unusually quiet afternoon                                                0.0   
decided learn new language year                                                  0.0   
rainy day make long nap enjoyable                                                0.0   
technology change faster people expect                                           0.0   
dog waited patiently door                                                        0.0   
early morning walk boost productivity                                            0.0   
forgot parked car                                                                0.0   
fresh

In [ ]:
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

def document_wordnet_similarity(doc1, doc2):
    tokens1 = word_tokenize(doc1.lower())
    tokens2 = word_tokenize(doc2.lower())

    synsets1 = [s for token in tokens1 for s in wordnet.synsets(token)]
    synsets2 = [s for token in tokens2 for s in wordnet.synsets(token)]

    if not synsets1 or not synsets2:
        return 0.0

    max_similarities = []
    for s1 in synsets1:
        max_sim_for_s1 = 0.0
        for s2 in synsets2:
            sim = s1.path_similarity(s2)
            if sim is not None and sim > max_sim_for_s1:
                max_sim_for_s1 = sim
        max_similarities.append(max_sim_for_s1)

    if max_similarities:
        return np.mean(max_similarities)
    else:
        return 0.0

documents = df['preprocessed_assignment_statements']
num_documents = len(documents)
wordnet_sim_matrix = np.zeros((num_documents, num_documents))

for i in range(num_documents):
    for j in range(num_documents):
        if i == j:
            wordnet_sim_matrix[i, j] = 1.0
        else:
            wordnet_sim_matrix[i, j] = document_wordnet_similarity(documents[i], documents[j])

df_wordnet_sim = pd.DataFrame(wordnet_sim_matrix, index=documents, columns=documents)

print("WordNet Similarity Matrix (Path Similarity):")
print(df_wordnet_sim)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


WordNet Similarity Matrix (Path Similarity):
preprocessed_assignment_statements           sun dipped horizon city light flickered  \
preprocessed_assignment_statements                                                     
sun dipped horizon city light flickered                                     1.000000   
cup coffee sometimes solve half day problem                                 0.176462   
library unusually quiet afternoon                                           0.229348   
decided learn new language year                                             0.261568   
rainy day make long nap enjoyable                                           0.231521   
technology change faster people expect                                      0.221571   
dog waited patiently door                                                   0.194268   
early morning walk boost productivity                                       0.199565   
forgot parked car                                                          